<a href="https://colab.research.google.com/github/msolowes-star/DI-Bootcamp/blob/main/Week16MiniProject1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project 1 — Scraping a Dynamic Webpage with Selenium and BeautifulSoup

**Developers Institute | Week 16**

## Project objective
Use Selenium to render a JavaScript-powered e-commerce page, BeautifulSoup to extract product names and prices, and pandas to save the results in a CSV file.

**Source:** [ScrapingCourse JavaScript Rendering Challenge](https://www.scrapingcourse.com/javascript-rendering)

**Source change:** The suggested InMotion Hosting site displayed a Cloudflare security verification page in automated Chrome, so it could not be scraped reliably. I used a public JavaScript-rendering practice page instead. This project extracts **product names and prices**, not hosting-plan features.

## 1. Install dependencies

Install the Python packages used for browser automation, HTML parsing, and data handling.

In [ ]:
!pip -q install selenium beautifulsoup4 pandas chromedriver-autoinstaller

## 2. Import libraries

In [ ]:
import shutil
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
import chromedriver_autoinstaller

## 3. Set up headless Chromium

Colab runs without a desktop browser window, so Chrome is launched in headless mode. The following installation steps match the working browser configuration from the original notebook.

In [ ]:
# Install non-Snap Chromium in this Colab environment.
!apt-get remove -y chromium-browser chromium-chromedriver > /dev/null 2>&1
!add-apt-repository -y ppa:xtradeb/apps > /dev/null 2>&1
!apt-get update -qq
!apt-get install -y chromium > /dev/null 2>&1

chrome_options = Options()
chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--window-size=1920,1080")

chrome_path = shutil.which("chromium") or shutil.which("chromium-browser")
if chrome_path:
    chrome_options.binary_location = chrome_path

driver_path = chromedriver_autoinstaller.install()
driver = webdriver.Chrome(
    service=Service(driver_path),
    options=chrome_options
)
print("Selenium WebDriver initialized successfully.")

## 4. Load the dynamic webpage

Wait until the JavaScript-rendered product elements appear, rather than relying on a fixed sleep duration.

In [ ]:
url = "https://www.scrapingcourse.com/javascript-rendering"
driver.get(url)

wait = WebDriverWait(driver, 30)
wait.until(lambda d: len(d.find_elements("css selector", "[data-testid='product-info']")) > 0)

html = driver.page_source
print("Page title:", driver.title)
print("Current URL:", driver.current_url)
print("Rendered HTML length:", len(html))

## 5. Parse rendered HTML with BeautifulSoup

The page uses `div.product-info` containers, with `span.product-name` and `span.product-price` for each product.

In [ ]:
soup = BeautifulSoup(html, "html.parser")
product_cards = soup.select("div.product-info")
print("Product cards found:", len(product_cards))

## 6. Extract product names and prices

Each product name is paired with its price from the same card. Empty or incomplete cards are skipped.

In [ ]:
products = []

for card in product_cards:
    name_element = card.select_one(".product-name")
    price_element = card.select_one(".product-price")

    if name_element and price_element:
        products.append({
            "Product Name": name_element.get_text(" ", strip=True),
            "Price": price_element.get_text(" ", strip=True)
        })

df_products = pd.DataFrame(products, columns=["Product Name", "Price"])

print("Products successfully extracted:", len(df_products))
assert not df_products.empty, "No products were extracted. Inspect the page or selectors."
display(df_products)

## 7. Validate and export the results

Check for missing values and save the scraped records to `dynamic_products.csv`.

In [ ]:
print("DataFrame shape:", df_products.shape)
print("Missing values by column:")
print(df_products.isna().sum())

df_products.to_csv("dynamic_products.csv", index=False, encoding="utf-8-sig")
print("Saved dynamic_products.csv")

## 8. Close the browser

Release the browser resources after the extraction is complete.

In [ ]:
driver.quit()
print("Selenium WebDriver closed successfully.")

## Conclusion

This mini-project demonstrated how to use **Selenium** to load JavaScript-rendered product information, **BeautifulSoup** to parse the rendered HTML, and **pandas** to structure and export the extracted records.

In the successful original run, the scraper extracted **12 products**, including product names and prices, and saved them as `dynamic_products.csv`.

**Limitation:** The originally suggested InMotion Hosting site could not be accessed through automated Chrome because it displayed a Cloudflare verification page. The alternative practice website provides product names and prices, but not hosting-plan features. The results are therefore a demonstration of the same scraping workflow using a different dataset.

**Skills practiced:** Selenium WebDriver, headless Chromium, JavaScript-rendered content, explicit waits, CSS selectors, BeautifulSoup, pandas DataFrames, CSV export, and browser cleanup.